# Práctica Pipelines - Dataset titanic

En el siguiente dataset intentaremos clasificar supervivientes del titanic en función de sus variables.

## Parte 1

Proceso el dataset; aplicando pipelines quite las columnas que considere, realice las imputaciones y/o escalamientos según corresponda y su conocimiento e intuicion le indiquen. Intente clasificar usando KNN. 


## Parte 2

Reutilizando lo anterior tanto como sea posible evalue utilizando otro clasificador. Pruebe distintas alternativas de transformaciones. Evalue que opción (transformaciones+clasificador) ofrece los mejores resultados.

### Dataset "Titanic"


In [5]:
import pandas as pd

In [9]:
df = pd.read_csv("https://raw.githubusercontent.com/PabloSoligo2014/3670-UNLaM-CD/refs/heads/main/datasets/titanic.csv")

In [11]:
df.head()

,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
0,1,0,3,"Braund, Mr. Owen Harris",male,22.0,1,0,A/5 21171,7.2500,NaN,S
1,2,1,1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",female,38.0,1,0,PC 17599,71.2833,C85,C
2,3,1,3,"Heikkinen, Miss. Laina",female,26.0,0,0,STON/O2. 3101282,7.9250,NaN,S
3,4,1,1,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",female,35.0,1,0,113803,53.1000,C123,S
4,5,0,3,"Allen, Mr. William Henry",male,35.0,0,0,373450,8.0500,NaN,S


In [12]:
df.isnull().sum()/len(df)*100

PassengerId     0.000000
Survived        0.000000
Pclass          0.000000
Name            0.000000
Sex             0.000000
Age            19.865320
SibSp           0.000000
Parch           0.000000
Ticket          0.000000
Fare            0.000000
Cabin          77.104377
Embarked        0.224467
dtype: float64

---
## Parte 1 — Pipeline + KNN

### Decisiones de preprocesamiento

| Columna | Acción | Motivo |
|---|---|---|
| `PassengerId` | Eliminar | ID sin valor predictivo |
| `Name` | Eliminar | Texto libre |
| `Ticket` | Eliminar | Código alfanumérico sin patrón claro |
| `Cabin` | Eliminar | 77 % de valores nulos |
| `Age` | Imputar mediana + escalar | 20 % nulos; mediana robusta a outliers |
| `Fare` | Escalar | Sin nulos pero distribución sesgada |
| `Pclass`, `SibSp`, `Parch` | Escalar | Numéricas sin nulos |
| `Sex`, `Embarked` | Imputar moda + OneHotEncoder | Categóricas; `Embarked` tiene 2 nulos |

In [ ]:
from sklearn.pipeline import Pipeline, make_pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.neighbors import KNeighborsClassifier
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.metrics import classification_report, ConfusionMatrixDisplay, accuracy_score
import matplotlib.pyplot as plt
import numpy as np

In [ ]:
# Analisis exploratorio

df.head(5)
import sweetviz as sv
reporte = sv.analyze(df, target_feat="Survived")
reporte.show_html("reporte.html")  # abre en el browser automáticamente
# AGE MISSING:177(20%)
# Cabin MISSING:687(77%)
# Embarked MISSING:2(0.2%) 

# Definir features y separar target
COLS_DROP   = ["PassengerId", "Name", "Ticket", "Cabin"]
NUM_FEATURES = ["Pclass", "Age", "SibSp", "Parch", "Fare"]
CAT_FEATURES = ["Sex", "Embarked"]

X = df.drop(columns=["Survived"] + COLS_DROP)
y = df["Survived"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f"Train: {X_train.shape} | Test: {X_test.shape}")
print(f"Distribución target (train):\n{y_train.value_counts(normalize=True).round(3)}")

                                             |          | [  0%]   00:00 -> (? left)

Report reporte.html was generated! NOTEBOOK/COLAB USERS: the web browser MAY not pop up, regardless, the report IS saved in your notebook/colab files.
Train: (712, 7) | Test: (179, 7)
Distribución target (train):
Survived
0    0.617
1    0.383
Name: proportion, dtype: float64


In [ ]:
knn_pipeline = Pipeline([
    ("prep", ColumnTransformer([
        ("num", Pipeline([
            ("imputer", SimpleImputer(strategy="median")),
            ("scaler",  StandardScaler()),
        ]), NUM_FEATURES),
        ("cat", Pipeline([
            ("imputer", SimpleImputer(strategy="most_frequent")),
            ("encoder", OneHotEncoder(handle_unknown="ignore")),
        ]), CAT_FEATURES),
    ])),
    ("clf", KNeighborsClassifier(n_neighbors=5)),
])

knn_pipeline

In [ ]:
from sklearn.model_selection import GridSearchCV

param_grid = {
    "clf__n_neighbors": range(1, 31),
    "clf__weights":     ["uniform", "distance"],
    "clf__metric":      ["euclidean", "manhattan"],
}

grid = GridSearchCV(knn_pipeline, param_grid, cv=5, scoring="accuracy")
grid.fit(X_train, y_train)

print(f"Mejores parámetros: {grid.best_params_}")
print(f"Mejor CV accuracy:  {grid.best_score_:.4f}")

In [ ]:
y_pred = grid.predict(X_test)

print("=== Classification Report ===")
print(classification_report(y_test, y_pred, target_names=["No sobrevivió", "Sobrevivió"]))

fig, ax = plt.subplots(figsize=(5, 4))
ConfusionMatrixDisplay.from_predictions(
    y_test, y_pred,
    display_labels=["No sobrevivió", "Sobrevivió"],
    ax=ax, colorbar=False
)
plt.title(f"KNN — {grid.best_params_}")
plt.tight_layout()
plt.show()

---
## Parte 2 — Otro clasificador, otras transformaciones

Se prueba **RandomForestClassifier**. Los árboles no son sensibles a la escala de las variables, por lo que el pipeline se simplifica: se elimina el `StandardScaler` en las numéricas. Las categóricas mantienen la imputación y el OHE.

In [ ]:
from sklearn.ensemble import RandomForestClassifier

# Pipeline simplificado: sin StandardScaler en numéricas (innecesario para árboles)
rf_pipeline = Pipeline([
    ("prep", ColumnTransformer([
        ("num", SimpleImputer(strategy="median"), NUM_FEATURES),
        ("cat", make_pipeline(SimpleImputer(strategy="most_frequent"), OneHotEncoder(handle_unknown="ignore")), CAT_FEATURES),
    ])),
    ("clf", RandomForestClassifier(n_estimators=100, random_state=42)),
])

rf_pipeline.fit(X_train, y_train)
y_pred_rf = rf_pipeline.predict(X_test)

print("=== RandomForest — Classification Report ===")
print(classification_report(y_test, y_pred_rf, target_names=["No sobrevivió", "Sobrevivió"]))

In [ ]:
from sklearn.preprocessing import MinMaxScaler

# Pipeline alternativo KNN: imputer mean + MinMaxScaler, K fijo sin GridSearch
knn_simple = Pipeline([
    ("prep", ColumnTransformer([
        ("num", make_pipeline(SimpleImputer(strategy="mean"), MinMaxScaler()), NUM_FEATURES),
        ("cat", make_pipeline(SimpleImputer(strategy="most_frequent"), OneHotEncoder(handle_unknown="ignore")), CAT_FEATURES),
    ])),
    ("clf", KNeighborsClassifier(n_neighbors=7)),
])

knn_simple.fit(X_train, y_train)
y_pred_simple = knn_simple.predict(X_test)

print("=== KNN simple (K=7, MinMaxScaler) — Classification Report ===")
print(classification_report(y_test, y_pred_simple, target_names=["No sobrevivió", "Sobrevivió"]))

In [ ]:
# Comparación final entre los tres modelos
best_k    = grid.best_params_["clf__n_neighbors"]
knn_label = f"KNN GridSearch (K={best_k}) + StandardScaler"

resultados = {
    knn_label:                              accuracy_score(y_test, y_pred),
    "KNN simple (K=7) + MinMaxScaler":      accuracy_score(y_test, y_pred_simple),
    "RandomForest + sin escalado":          accuracy_score(y_test, y_pred_rf),
}

print("=== Comparación de accuracy en test ===")
for nombre, acc in sorted(resultados.items(), key=lambda x: x[1], reverse=True):
    print(f"  {acc:.4f}  {nombre}")

fig, axes = plt.subplots(1, 3, figsize=(14, 4))
for ax, (preds, titulo) in zip(axes, [
    (y_pred,        f"KNN GridSearch K={best_k}"),
    (y_pred_simple, "KNN simple K=7"),
    (y_pred_rf,     "RandomForest"),
]):
    ConfusionMatrixDisplay.from_predictions(
        y_test, preds,
        display_labels=["No sobrevivió", "Sobrevivió"],
        ax=ax, colorbar=False
    )
    ax.set_title(titulo)
plt.tight_layout()
plt.show()

In [ ]:
# Comparación final entre ambos modelos
from sklearn.metrics import accuracy_score

best_k    = grid.best_params_["clf__n_neighbors"]
knn_label = f"KNN (K={best_k}, w={grid.best_params_['clf__weights']}, m={grid.best_params_['clf__metric']}) + StandardScaler"

resultados = {
    knn_label:                       accuracy_score(y_test, y_pred),
    "RandomForest + sin escalado":   accuracy_score(y_test, y_pred_rf),
}

print("=== Comparación de accuracy en test ===")
for nombre, acc in sorted(resultados.items(), key=lambda x: x[1], reverse=True):
    print(f"  {acc:.4f}  {nombre}")

fig, axes = plt.subplots(1, 2, figsize=(10, 4))
for ax, (preds, titulo) in zip(axes, [(y_pred, f"KNN K={best_k}"), (y_pred_rf, "RandomForest")]):
    ConfusionMatrixDisplay.from_predictions(
        y_test, preds,
        display_labels=["No sobrevivió", "Sobrevivió"],
        ax=ax, colorbar=False
    )
    ax.set_title(titulo)
plt.tight_layout()
plt.show()